# Name1: Bereinigte State-of-the-Art Variante

Dieses Notebook setzt alle Ziele robust um:
1. Ein modernes LLM laden (effizientes Instruct-Modell).
2. Kleine Trainingsdaten vorbereiten.
3. Innerhalb von maximal 5 Minuten trainieren.
4. Eine Beispielabfrage mit dem trainierten Modell ausführen.


## 1) Abhängigkeiten installieren

Hinweis: In Jupyter bitte diese Zelle bei Bedarf einmal ausführen.


In [1]:
%pip install -q -U transformers datasets accelerate peft bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 13.4 MB/s eta 0:00:00


## 2) Imports und Konfiguration


In [2]:
import time
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model

SEED = 42
torch.manual_seed(SEED)

MODEL_ID = "HuggingFaceTB/SmolLM2-360M-Instruct"
MAX_LENGTH = 256
TIME_LIMIT_SECONDS = 5 * 60
OUTPUT_DIR = "./artifacts/name1_sota"

use_cuda = torch.cuda.is_available()
print("CUDA verfügbar:", use_cuda)


CUDA verfügbar: False


## 3) Trainingsdaten vorbereiten

Kleine, synthetische Instruction-Daten für schnelles Fine-Tuning.


In [3]:
examples = [
    {"instruction": "Erkläre kurz, was Overfitting ist.", "response": "Overfitting bedeutet, dass ein Modell Trainingsdaten zu genau lernt und auf neuen Daten schlechter generalisiert."},
    {"instruction": "Nenne zwei Vorteile von Unit-Tests.", "response": "Unit-Tests erkennen Regressionen früh und erleichtern Refactoring."},
    {"instruction": "Was ist ein Tokenizer?", "response": "Ein Tokenizer zerlegt Text in verarbeitbare Einheiten, sogenannte Tokens."},
    {"instruction": "Formuliere eine höfliche Absage für ein Meeting.", "response": "Vielen Dank für die Einladung. Leider kann ich am vorgeschlagenen Termin nicht teilnehmen."},
    {"instruction": "Schreibe eine knappe Definition von REST.", "response": "REST ist ein Architekturprinzip für Web-APIs mit klaren Ressourcen und HTTP-Methoden."},
    {"instruction": "Wozu dient Gradient Descent?", "response": "Gradient Descent minimiert eine Verlustfunktion durch schrittweise Anpassung der Modellparameter."},
    {"instruction": "Wie verbessert man Prompt-Qualität?", "response": "Durch klare Ziele, konkrete Formatvorgaben und relevante Kontextinformationen."},
    {"instruction": "Nenne einen Unterschied zwischen Precision und Recall.", "response": "Precision misst die Genauigkeit positiver Vorhersagen, Recall die Vollständigkeit erkannter positiver Fälle."},
    {"instruction": "Was macht ein Lernraten-Scheduler?", "response": "Er passt die Lernrate während des Trainings an, um Stabilität und Konvergenz zu verbessern."},
    {"instruction": "Erkläre kurz Embeddings.", "response": "Embeddings sind dichte Vektordarstellungen, die semantische Ähnlichkeiten von Texten abbilden."},
]

raw_dataset = Dataset.from_list(examples)
print(raw_dataset)


Dataset({
    features: ['instruction', 'response'],
    num_rows: 10
})


## 4) Modell und Tokenizer laden (optional 4-bit)

Bei CUDA wird automatisch versucht, quantisiert zu laden. Falls `bitsandbytes` nicht verfügbar ist, wird sauber auf Standard-Laden zurückgefallen.


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

quant_config = None
if use_cuda:
    try:
        import bitsandbytes  # noqa: F401
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
        print("4-bit Quantisierung aktiviert.")
    except Exception as e:
        print("bitsandbytes nicht nutzbar, lade ohne 4-bit:", e)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto" if use_cuda else None,
    torch_dtype=torch.bfloat16 if use_cuda else torch.float32,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "up_proj", "down_proj", "gate_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/724M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

trainable params: 8,683,520 || all params: 370,504,640 || trainable%: 2.3437


## 5) Tokenisierung und Trainings-Setup


In [5]:
def format_example(example):
    if tokenizer.chat_template:
        messages = [
            {"role": "user", "content": example["instruction"]},
            {"role": "assistant", "content": example["response"]},
        ]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
    else:
        text = f"### Instruction:
{example['instruction']}

### Response:
{example['response']}"
    return {"text": text}

formatted_dataset = raw_dataset.map(format_example)

def tokenize_fn(batch):
    tokenized = tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

train_dataset = formatted_dataset.map(tokenize_fn, batched=True, remove_columns=formatted_dataset.column_names)
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

print(train_dataset)


SyntaxError: unterminated f-string literal (detected at line 13) (1374766932.py, line 13)

## 6) Training mit 5-Minuten-Limit


In [ ]:
class TimeLimitCallback(TrainerCallback):
    def __init__(self, max_seconds):
        self.max_seconds = max_seconds
        self.start_time = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()

    def on_step_end(self, args, state, control, **kwargs):
        if self.start_time and (time.time() - self.start_time) > self.max_seconds:
            control.should_training_stop = True
            print(f"Zeitlimit von {self.max_seconds} Sekunden erreicht. Training wird gestoppt.")
        return control

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=5,
    max_steps=120,
    logging_steps=5,
    save_steps=200,
    fp16=use_cuda,
    bf16=False,
    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collator,
    callbacks=[TimeLimitCallback(TIME_LIMIT_SECONDS)],
)

start = time.time()
train_result = trainer.train()
elapsed = time.time() - start
print(f"Training abgeschlossen in {elapsed:.1f} Sekunden")
print(train_result)


## 7) Beispielabfrage mit dem trainierten Modell


In [ ]:
model.eval()

prompt = "Erkläre in 3 Sätzen, warum Datenqualität für Machine Learning entscheidend ist."
if tokenizer.chat_template:
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )
else:
    prompt_text = f"### Instruction:
{prompt}

### Response:
"

inputs = tokenizer(prompt_text, return_tensors="pt")
if not hasattr(model, "hf_device_map"):
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )

response = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(response)


## 8) Ergebnis

Dieses Notebook ist die bereinigte Zielversion ohne Retry-Duplikate und mit stabiler Pipeline:
- Laden eines modernen LLMs
- Datenaufbereitung
- Fine-Tuning mit hartem Zeitlimit (5 Minuten)
- Beispielinferenz
